# Golden Truth Generation — Difficulty Label Extraction

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

---

This notebook generates **golden truth difficulty labels** for the Coursera dataset  
using Google Gemini as a proxy annotator.

**Sections**
- Cell 1: Install dependencies
- Cell 2: Generate golden truth difficulty labels and export to CSV

**Disclosure note for methodology:**  
Golden truth difficulty labels are generated by Google Gemini (a model not used in any  
experiment being evaluated), serving as a silver-standard proxy for human annotation.

---

**Setup**
- Step 1: Run Cell 1 to install dependencies. Restart runtime if prompted.
- Step 2: Add your Gemini API key as a Colab Secret named `GEMINI_API_KEY`.
- Step 3: Run Cell 2.


In [ ]:
# =============================================================================
# CELL 1 — INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

PACKAGES = [
    ("google-generativeai>=0.5.0",),
    ("pandas>=2.0.0",),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


for pkg_group in PACKAGES:
    pip_install(*pkg_group)

print("All packages installed successfully.")


All packages installed successfully.


In [ ]:
# =============================================================================
# CELL 2 — DIFFICULTY LABEL GOLDEN TRUTH GENERATION
#
# Sections
# --------
#   A.  Imports
#   B.  Configuration
#   C.  Dataset load
#   D.  API key setup  (supports up to 3 keys with automatic fallback)
#   E.  Prompt template
#   F.  Gemini caller
#   G.  Generation loop
#   H.  Quality checks
#   I.  Save & download
# =============================================================================


# =============================================================================
# A.  IMPORTS
# =============================================================================

import getpass
import html
import os
import random
import re
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import google.generativeai as genai
import pandas as pd
from google.colab import drive

try:
    from google.colab import files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

SEPARATOR_SM: int = 40
SEPARATOR_MD: int = 55
SEPARATOR_LG: int = 65


# =============================================================================
# B.  CONFIGURATION
# =============================================================================

# Gemini model to use as proxy annotator.
# Must NOT be any model used in the main experiment.
GEMINI_MODEL: str = "gemini-2.5-flash"

# Generation parameters
TEMPERATURE: float = 0.2   # Low temperature for consistent, factual output
TOP_P: float = 0.9
MAX_OUTPUT_TOKENS: int = 64  # Difficulty label is a single word — short output needed

# Inter-call delay to respect API rate limits (seconds)
DELAY_MIN: float = 8.0
DELAY_MAX: float = 12.0

# Retry settings for API calls
MAX_RETRIES: int = 3
RETRY_BACKOFF_MIN: float = 10.0
RETRY_BACKOFF_MAX: float = 20.0

# Max retries if response is not a valid difficulty label
MAX_GENERATION_RETRIES: int = 2  # Including the initial attempt, so 2 retries means 3 total attempts

# Dataset
N_ROWS: int = 25
# Names of the three Colab Secrets — add each key under the matching name
GEMINI_SECRET_NAMES: List[str] = [
    "GEMINI_API_KEY",    # Key 1 (primary)
    "GEMINI_API_KEY_2",  # Key 2 (first fallback)
    "GEMINI_API_KEY_3",  # Key 3 (second fallback)
]

# Valid difficulty labels the model must choose from
VALID_DIFFICULTY_LABELS: List[str] = ["Beginner", "Intermediate", "Advanced"]


def print_config() -> None:
    """Print the current generation configuration."""
    total_calls = N_ROWS * (MAX_GENERATION_RETRIES + 1)  # Max possible calls
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Golden Truth Difficulty Label Extraction — Configuration")
    print("-" * SEPARATOR_SM)
    print(f"  Gemini model       : {GEMINI_MODEL}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  top_p              : {TOP_P}")
    print(f"  Max output tokens  : {MAX_OUTPUT_TOKENS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Max API retries    : {MAX_RETRIES}")
    print(f"  Max gen retries    : {MAX_GENERATION_RETRIES}")
    print(f"  Est. total calls   : {total_calls} (max possible)")
    print(f"  Est. time (max)    : ~{avg_delay_min:.1f} min")
    print(f"  Valid labels       : {VALID_DIFFICULTY_LABELS}")
    print(f"  API keys available : {len(GEMINI_SECRET_NAMES)}")


print_config()


# =============================================================================
# C.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load and clean the Coursera CSV dataset from Google Drive.

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns.

    Raises:
        FileNotFoundError: If the dataset CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)

    drive.mount("/content/drive")

    dataset_path = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    print(f"  Loaded {len(df)} courses successfully.")
    return df


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)


# =============================================================================
# D.  API KEY SETUP  (up to 3 keys, automatic quota-based fallback)
# =============================================================================

def load_api_key(secret_name: str, key_number: int) -> str:
    """Load a single Gemini API key from Colab Secrets or manual input.

    Args:
        secret_name: The Colab Secret key name (e.g. "GEMINI_API_KEY").
        key_number:  Human-readable key number (1, 2, or 3) for prompts.

    Returns:
        API key string, or empty string if the key was not provided.
    """
    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Key {key_number} loaded from Colab Secret '{secret_name}'.")
        except Exception:
            pass  # Key not set — will try manual entry for key 1 only

    if not api_key and key_number == 1:
        api_key = getpass.getpass(
            f"  Enter your primary Gemini API key ('{secret_name}'): "
        ).strip()

    if api_key:
        print(f"  Key {key_number} : AIza...{api_key[-4:]}")
    else:
        print(f"  Key {key_number} : not set (secret '{secret_name}' not found — skipped).")

    return api_key


def load_all_api_keys(secret_names: List[str]) -> List[str]:
    """Load all configured API keys, returning only non-empty ones.

    Args:
        secret_names: Ordered list of Colab Secret names to try.

    Returns:
        List of non-empty API key strings, in priority order.

    Raises:
        ValueError: If no valid key is found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    keys: List[str] = []
    for i, name in enumerate(secret_names, start=1):
        key = load_api_key(name, i)
        if key:
            keys.append(key)

    if not keys:
        raise ValueError(
            "No Gemini API key provided. "
            f"Add at least one key as a Colab Secret named '{secret_names[0]}' "
            "or enter it when prompted."
        )

    print(f"\n  {len(keys)} key(s) available for automatic fallback.")
    return keys


def build_gemini_client(api_key: str) -> genai.GenerativeModel:
    """Configure the genai library with the given key and return a client.

    Args:
        api_key: Gemini API key string.

    Returns:
        Configured GenerativeModel instance.
    """
    genai.configure(api_key=api_key)
    return genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=genai.GenerationConfig(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
    )


ALL_API_KEYS: List[str] = load_all_api_keys(GEMINI_SECRET_NAMES)

_active_key_index: int = 0
GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
print(f"  Gemini client ready : {GEMINI_MODEL}  (using key 1 of {len(ALL_API_KEYS)})")


def rotate_api_key() -> bool:
    """Switch to the next available API key and rebuild the Gemini client.

    Returns:
        True if rotation succeeded, False if all keys are exhausted.
    """
    global _active_key_index, GEMINI_CLIENT

    next_index = _active_key_index + 1
    if next_index >= len(ALL_API_KEYS):
        return False

    _active_key_index = next_index
    GEMINI_CLIENT = build_gemini_client(ALL_API_KEYS[_active_key_index])
    print(
        f"\n  *** Rotated to API key {_active_key_index + 1} of {len(ALL_API_KEYS)} ***\n"
    )
    return True


# =============================================================================
# E.  PROMPT TEMPLATE
# =============================================================================

def build_difficulty_label_prompt(description: str) -> str:
    """Build the golden truth difficulty label prompt for a single course.

    The prompt instructs Gemini to infer the difficulty level solely from
    the course description text, returning a single label from the
    controlled vocabulary.

    Args:
        description: Cleaned course description string.

    Returns:
        Fully rendered prompt string.
    """
    labels_str = ", ".join(VALID_DIFFICULTY_LABELS)
    return f"""Act as an objective academic content classifier. Your task is to determine the difficulty level of a course based on its description.

    ### Evaluation Criteria:
    - Beginner: No prior knowledge needed; introduces foundational concepts.
    - Intermediate: Requires some prior knowledge or domain experience.
    - Advanced: Targets experienced practitioners; covers complex or specialized topics.

    ### Instruction:
    Analyze the description below. Select exactly one label from the provided options: {labels_str}.
    Output only the label and nothing else. If the description is ambiguous, select the most likely fit based on the criteria.

    ### Course Description to Evaluate:
    {description}

    ### Difficulty Level:
    """


# =============================================================================
# F.  GEMINI CALLER
# =============================================================================

QUOTA_ERROR_PHRASES: Tuple[str, ...] = (
    "quota",
    "rate limit",
    "resource exhausted",
    "429",
)


class QuotaExceeded(Exception):
    """Raised when ALL available Gemini API keys have hit their quota."""


def is_quota_error(exc: Exception) -> bool:
    """Return True if the exception signals an API quota error."""
    return any(phrase in str(exc).lower() for phrase in QUOTA_ERROR_PHRASES)


def call_gemini(
    prompt: str,
    retries: int = MAX_RETRIES,
) -> Tuple[str, float]:
    """Call the Gemini API and return (response_text, latency_seconds)."""
    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"    Waiting {delay:.1f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = GEMINI_CLIENT.generate_content(prompt)
            latency = round(time.perf_counter() - t_start, 3)
            text = response.text.strip() if response.text else ""
            print(f"OK ({latency}s)")
            return text, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)

            if "503" in str(exc) or "service unavailable" in str(exc).lower():
                print(f"\n    503 Service Unavailable (attempt {attempt}/{retries}). Retrying same key...")
                if attempt < retries:
                    backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                    print(f"    Retrying in {backoff:.1f}s...")
                    time.sleep(backoff)
                    continue
                else:
                    print("    All retries exhausted on 503. Recording empty response.")
                    return "", latency

            if is_quota_error(exc):
                print(f"\n    Quota limit hit on key {_active_key_index + 1}.")
                rotated = rotate_api_key()
                if not rotated:
                    raise QuotaExceeded(
                        f"All {len(ALL_API_KEYS)} API key(s) have hit their quota. "
                        "Original error: " + str(exc)
                    ) from exc
                print(f"    Retrying with key {_active_key_index + 1}...")
                delay2 = random.uniform(DELAY_MIN, DELAY_MAX)
                print(f"    Waiting {delay2:.1f}s...", end=" ", flush=True)
                time.sleep(delay2)
                attempt = 0
                continue

            print(f"\n    Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"    Retrying in {backoff:.1f}s...")
                time.sleep(backoff)
            else:
                print("    All retries exhausted. Recording empty response.")
                return "", latency


# =============================================================================
# G.  GENERATION LOOP
# =============================================================================

def parse_difficulty(raw_text: str) -> str:
    """Normalise model output to a valid difficulty label.

    Strips whitespace and punctuation, then does a case-insensitive match
    against VALID_DIFFICULTY_LABELS.

    Args:
        raw_text: Raw model output string.

    Returns:
        Matched label from VALID_DIFFICULTY_LABELS, or empty string if
        no match is found.
    """
    cleaned = raw_text.strip().strip('"\'.,:;').strip()
    for label in VALID_DIFFICULTY_LABELS:
        if cleaned.lower() == label.lower():
            return label
    return ""


def build_gold_record(
    idx: int,
    row: pd.Series,
    gold_difficulty: str,
    raw_response: str,
    latency: float,
    status: str,
) -> Dict:
    """Assemble a single gold difficulty record dict.

    Args:
        idx:             Row index in the dataset.
        row:             Dataset row as a Series.
        gold_difficulty: Normalised difficulty label string.
        raw_response:    Raw model output before parsing.
        latency:         API call latency in seconds.
        status:          "ok", "empty", or "invalid_label".

    Returns:
        Dictionary with all gold record fields.
    """
    return {
        "course_idx":           idx,
        "course_title":         str(row.get("course_title", "Unknown")),
        "course_organization":  str(row.get("course_organization", "N/A")),
        "original_difficulty":  str(row.get("course_difficulty", "N/A")),
        "description":          str(row.get("description", "")),
        "gold_difficulty":      gold_difficulty,
        "raw_response":         raw_response,
        "latency_seconds":      latency,
        "status":               status,
        "gemini_model":         GEMINI_MODEL,
        "api_key_index":        _active_key_index + 1,
        "temperature":          TEMPERATURE,
        "timestamp":            datetime.now(timezone.utc).isoformat(),
    }


gold_records: List[Dict] = []
quota_hit: bool = False
generation_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("GENERATING GOLDEN TRUTH DIFFICULTY LABELS")
print("=" * SEPARATOR_LG)

try:
    for idx, row in DATASET_DF.iterrows():
        title = str(row.get("course_title", "Unknown"))
        description = str(row.get("description", ""))

        print(f"  [{idx:02d}] {title[:60]}...")

        prompt = build_difficulty_label_prompt(description)

        current_difficulty: str = ""
        current_raw: str = ""
        current_latency: float = 0.0
        current_status: str = "failed_invalid_label"

        for attempt in range(MAX_GENERATION_RETRIES + 1):
            if attempt > 0:
                print(f"    Retrying generation (attempt {attempt + 1}/{MAX_GENERATION_RETRIES + 1}) for 'invalid_label'...")
                time.sleep(random.uniform(2.0, 5.0))

            raw_response, latency = call_gemini(prompt)
            difficulty = parse_difficulty(raw_response)

            if not raw_response:
                status = "empty"
            elif not difficulty:
                status = "invalid_label"
            else:
                status = "ok"

            current_difficulty = difficulty
            current_raw = raw_response
            current_latency = latency
            current_status = status

            if status == "ok":
                break

        gold_records.append(
            build_gold_record(idx, row, current_difficulty, current_raw, current_latency, current_status)
        )

except QuotaExceeded as exc:
    quota_hit = True
    print("\n" + "!" * SEPARATOR_LG)
    print("*** ALL API KEYS QUOTA EXCEEDED — generation stopped here ***")
    print(f"  Records collected : {len(gold_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)

GOLD_DF: pd.DataFrame = pd.DataFrame(gold_records)
generation_status = "STOPPED EARLY (all keys quota exhausted)" if quota_hit else "COMPLETE"
print(f"\nGeneration {generation_status}. Total records: {len(GOLD_DF)}")


# =============================================================================
# H.  QUALITY CHECKS
# =============================================================================

def run_quality_checks(gold_df: pd.DataFrame) -> None:
    """Print a quality report on the generated gold difficulty labels.

    Checks performed:
    - Status distribution (ok / empty / invalid_label)
    - Label distribution across valid categories
    - Agreement rate between gold label and original dataset label
    - Latency summary

    Args:
        gold_df: DataFrame produced by the generation loop.
    """
    if gold_df.empty:
        print("No records to check.")
        return

    print("\n" + "=" * SEPARATOR_LG)
    print("QUALITY REPORT")
    print("=" * SEPARATOR_LG)

    # Status distribution
    print("\n  Status distribution")
    print("  " + "-" * SEPARATOR_SM)
    for status, count in gold_df["status"].value_counts().items():
        pct = count / len(gold_df) * 100
        print(f"    {status:<20} {count:>3}  ({pct:.1f}%)")

    # API key usage breakdown
    if "api_key_index" in gold_df.columns:
        print("\n  API key usage")
        print("  " + "-" * SEPARATOR_SM)
        for key_idx, count in gold_df["api_key_index"].value_counts().sort_index().items():
            print(f"    Key {key_idx} : {count} record(s)")

    # Label distribution
    ok_df = gold_df[gold_df["status"] == "ok"]
    if not ok_df.empty:
        print("\n  Gold difficulty label distribution (status=ok)")
        print("  " + "-" * SEPARATOR_SM)
        for label in VALID_DIFFICULTY_LABELS:
            count = (ok_df["gold_difficulty"] == label).sum()
            pct = count / len(ok_df) * 100
            print(f"    {label:<15} {count:>3}  ({pct:.1f}%)")

    # Agreement with original dataset labels
    if not ok_df.empty and "original_difficulty" in ok_df.columns:
        print("\n  Agreement with original dataset labels (status=ok)")
        print("  " + "-" * SEPARATOR_SM)
        agree = (
            ok_df["gold_difficulty"].str.lower() ==
            ok_df["original_difficulty"].str.lower()
        ).sum()
        agree_pct = agree / len(ok_df) * 100
        print(f"    Matching labels : {agree} / {len(ok_df)}  ({agree_pct:.1f}%)")
        print("    Note: Disagreements indicate the model inferred a different level")
        print("          from the description — review these rows manually.")

        # List disagreements
        disagree_df = ok_df[
            ok_df["gold_difficulty"].str.lower() != ok_df["original_difficulty"].str.lower()
        ]
        if not disagree_df.empty:
            print(f"\n  Disagreements ({len(disagree_df)} rows)")
            print("  " + "-" * SEPARATOR_SM)
            for _, drow in disagree_df.iterrows():
                print(
                    f"    [{drow['course_idx']:02d}] {str(drow['course_title'])[:45]:<45} "
                    f"orig={drow['original_difficulty']:<15} gold={drow['gold_difficulty']}"
                )

    # Latency summary
    lat = gold_df["latency_seconds"]
    print("\n  Latency (seconds)")
    print("  " + "-" * SEPARATOR_SM)
    print(f"    Min    : {lat.min():.3f}s")
    print(f"    Max    : {lat.max():.3f}s")
    print(f"    Mean   : {lat.mean():.3f}s")
    print(f"    Total  : {lat.sum():.1f}s")

    print("\n" + "=" * SEPARATOR_LG)


run_quality_checks(GOLD_DF)


# =============================================================================
# I.  SAVE & DOWNLOAD
# =============================================================================

GOLD_FILENAME: str = f"gold_difficulty_coursera.csv"
FLAGGED_FILENAME: str = f"gold_difficulty_flagged_coursera.csv"


def save_gold_difficulty(
    gold_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save the gold difficulty DataFrame to CSV and optionally trigger downloads.

    Saves two files:
    - Full output: all records including failures.
    - Flagged output: only records with status != 'ok', for manual review.

    Args:
        gold_df:  DataFrame produced by the generation loop.
        in_colab: True if running inside Google Colab.
    """
    gold_df.to_csv(GOLD_FILENAME, index=False)
    print(f"\nSaved: {GOLD_FILENAME}  ({len(gold_df)} rows)")

    flagged_df = gold_df[gold_df["status"] != "ok"]
    if not flagged_df.empty:
        flagged_df.to_csv(FLAGGED_FILENAME, index=False)
        print(
            f"Saved: {FLAGGED_FILENAME}  ({len(flagged_df)} rows — manual review needed)"
        )
    else:
        print("No flagged records — all difficulty labels passed status checks.")

    if in_colab:
        print("\nDownloading files...")
        files.download(GOLD_FILENAME)
        if not flagged_df.empty:
            files.download(FLAGGED_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")

save_gold_difficulty(GOLD_DF, IN_COLAB)


Golden Truth Difficulty Label Extraction — Configuration
----------------------------------------
  Gemini model       : gemini-2.5-flash
  Temperature        : 0.2
  top_p              : 0.9
  Max output tokens  : 64
  Courses            : 25
  Max API retries    : 3
  Max gen retries    : 2
  Est. total calls   : 75 (max possible)
  Est. time (max)    : ~12.5 min
  Valid labels       : ['Beginner', 'Intermediate', 'Advanced']
  API keys available : 3

----------------------------------------
Dataset Load
----------------------------------------
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Found dataset at: /content/drive/MyDrive/research/data/coursera_clean_sample.csv
  Loaded 25 courses successfully.

----------------------------------------
API Key Setup
----------------------------------------
  Key 1 loaded from Colab Secret 'GEMINI_API_KEY'.
  Key 1 : AIza...km0M
  Key 2 loaded from Colab Secre


    Quota limit hit on key 1.

  *** Rotated to API key 2 of 3 ***

    Retrying with key 2...
    Waiting 9.6s... OK (1.853s)
  [13] introduction to data analytics for accounting professionals...
    Waiting 10.3s... 
    Attempt 1/3 failed: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 2.
    Retrying in 19.1s...

    Attempt 2/3 failed: Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 2.
    Retrying in 19.0s...
OK (1.699s)
  [14] statistics in psychological research...
    Waiting 8.1s... OK (1.751s)
  [15] plastic electronics...
    Waiting 8.6s... OK (1.446s)
  [16] learn english beginning grammar specialization...
    Waiting 9.8s... OK (1.6

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
